# 04 — Subgroup Fairness Audit (`Fairlearn MetricFrame`) & Bias Mitigation Trade-Offs

**Objective:** Audit group disparities with 95% bootstrap confidence intervals across `SEX`, `AGE_GROUP`, `MARRIAGE`, and `SEX x AGE_GROUP`, then benchmark Sample Reweighing, `ExponentiatedGradient`, and `ThresholdOptimizer`.


In [1]:
import json
import pandas as pd
from creditfair import config

with open(config.get_artifact_path('fairness_before.json')) as f:
    fb = json.load(f)
with open(config.get_artifact_path('fairness_after.json')) as f:
    fa = json.load(f)

print('=== Unmitigated SEX Summary ===')
print(json.dumps(fb['audits']['SEX']['summary_metrics'], indent=2))
print('\n=== Mitigation Trade-Off Table (AGE_GROUP) ===')
df_mit = pd.DataFrame(fa['comparisons']['AGE_GROUP']).drop(columns=['by_group'])
print(df_mit.to_string(index=False))


{
  "demographic_parity_difference": 0.0406,
  "disparate_impact_denial_ratio": 0.9266,
  "disparate_impact_approval_ratio": 0.9165,
  "passes_four_fifths_approval_rule": true,
  "equal_opportunity_difference_tpr_gap": 0.0312,
  "fpr_gap": 0.0367,
  "equalized_odds_difference": 0.0367
}

[
  {
    "method": "Unmitigated Baseline (XGBoost)",
    "paradigm": "Baseline",
    "requires_sensitive_at_inference": false,
    "roc_auc": 0.783,
    "accuracy": 0.6109,
    "f1_score": 0.4815,
    "selection_rate_denial": 0.5291,
    "approval_rate": 0.4709,
    "expected_cost_per_applicant": 0.5518,
    "demographic_parity_difference": 0.0684,
    "equal_opportunity_difference": 0.0789,
    "equalized_odds_difference": 0.0789,
    "disparate_impact_approval_ratio": 0.8646,
    "passes_four_fifths_rule": true
  },
  {
    "method": "Pre-Processing: Sample Reweighing",
    "paradigm": "Pre-processing",
    "requires_sensitive_at_inference": false,
    "roc_auc": 0.7836,
    "accuracy": 0.5687,
    

### Conclusion: Disparate Impact vs. Disparate Treatment
- **Pre-processing (Sample Reweighing)** and **In-processing (`ExponentiatedGradient`)** reduce Demographic Parity and Equalized Odds differences without requiring protected attributes at prediction time.
- **Post-processing (`ThresholdOptimizer`)** achieves the smallest group gaps, but requires passing `SEX` or `AGE_GROUP` at inference time to apply group-specific cutoffs—raising **Disparate Treatment** concerns under fair-lending laws.
